# Clean the Acebes et al. project-risk Excel dataset

This notebook takes the supplementary Excel file from *Beyond probability-impact matrices in project risk management: A quantitative methodology for risk prioritisation* and creates a clearer two-sheet workbook.

The output workbook has:

1. **Activities**: one row per activity, in order, with predecessors and connected schedule/cost risk IDs.
2. **Risks**: one row per risk ID, with schedule/cost type, affected activity, probability distribution, impact/severity distribution, and the original uncertainty bounds.

The notebook keeps the uncertainty information from the source file. Activity durations are kept as triangular distributions, and risk probability/impact rows are kept as uniform distributions with their min/max bounds.

## Important modelling note

The source Excel stores the planned activity network together with additional pseudo-activities for schedule risks (`A33:A41`). To make the baseline activity sheet readable, the notebook uses the predecessor relationships from the published activity table for the planned network. The risk table still keeps the raw activity field found in the source Excel, so that this information is not lost.

In [ ]:
from pathlib import Path
import re
import json
import pandas as pd

INPUT = Path('/mnt/data/41599_2024_3180_MOESM1_ESM(1).xlsx')  # Change this path if your Excel file has another name
OUTPUT_XLSX = Path('/mnt/data/cleaned_project_risk_dataset.xlsx')

# Activity predecessor list from Table 2 in the published paper. The raw Excel adjacency matrix
# contains inserted pseudo-activities A33:A41 for schedule risks, so this list is used for the
# baseline planned network.
TABLE2_PREDECESSORS = {
    'A1': 'Ai',
    'A2': 'A1',
    'A3': 'A1',
    'A4': 'A2, A3',
    'A5': 'A2',
    'A6': 'A2',
    'A7': 'A1',
    'A8': 'A5',
    'A9': 'A5',
    'A10': 'A6',
    'A11': 'A7',
    'A12': 'A7',
    'A13': 'A7',
    'A14': 'A11, A13',
    'A15': 'A14',
    'A16': 'A1',
    'A17': 'A2',
    'A18': 'A17',
    'A19': 'A3',
    'A20': 'A3, A4',
    'A21': 'A18',
    'A22': 'A21',
    'A23': 'A18',
    'A24': 'A21',
    'A25': 'A8',
    'A26': 'A21, A25',
    'A27': 'A9, A22, A23',
    'A28': 'A10, A24',
    'A29': 'A12, A20',
    'A30': 'A15, A16, A29',
    'A31': 'A26, A30',
    'A32': 'A27, A28, A31',
}

PHASE_BY_NUMBER = {
    **{i: 'Engineering' for i in range(1, 8)},
    **{i: 'Procurement' for i in range(8, 17)},
    **{i: 'Construction' for i in range(17, 33)},
}

COST_RISK_ID_MAP = {
    'RC_1': 'R10',
    'RC_2': 'R11',
    'RC_3': 'R12',
    'RC_4': 'R13',
    'RC_5': 'R14',
    'RC_6': 'R15',
}

PROBABILITY_LEVEL_INTERVAL = {
    'VL': '0-0.03',
    'L': '0.03-0.10',
    'M': '0.10-0.30',
    'H': '0.30-0.50',
    'VH': '0.50-1.00',
}

SEVERITY_LABEL_NOTE = {
    'VL': 'Very low',
    'L': 'Low',
    'M': 'Medium',
    'H': 'High',
    'B': 'Bajo / Low in original Spanish scale',
    'A': 'Alto / High in original Spanish scale',
    '-': 'No impact of this type specified',
}


def clean_value(x):
    if pd.isna(x):
        return None
    if isinstance(x, float) and x.is_integer():
        return int(x)
    return x


def parse_activity_number(activity_id):
    m = re.fullmatch(r'A(\d+)', str(activity_id))
    return int(m.group(1)) if m else None


def split_interval(text):
    if text is None or pd.isna(text) or str(text).strip() in {'', '-'}:
        return (None, None)
    parts = str(text).replace('–', '-').split('-')
    if len(parts) == 2:
        try:
            return (float(parts[0]), float(parts[1]))
        except ValueError:
            return (parts[0].strip(), parts[1].strip())
    return (None, None)


def load_source_workbook(path: Path):
    return {
        'Data': pd.read_excel(path, sheet_name='Data', header=None),
        'Risk Id': pd.read_excel(path, sheet_name='Risk Id', header=None),
    }


def extract_raw_activity_table(data):
    rows = []
    # Excel rows 7:38 in the source correspond to the 32 real activities A1:A32.
    for excel_row in range(7, 40):
        r = data.iloc[excel_row - 1]
        activity_id = clean_value(r[0])
        if not isinstance(activity_id, str) or not re.fullmatch(r'A\d+', activity_id):
            continue
        n = parse_activity_number(activity_id)
        rows.append({
            'Activity_ID': activity_id,
            'Activity_Number': n,
            'Phase': PHASE_BY_NUMBER.get(n),
            'Predecessors': TABLE2_PREDECESSORS.get(activity_id, clean_value(r[20])),
            'Duration_Distribution': clean_value(r[1]),
            'Duration_Model_Code': clean_value(r[2]),
            'Duration_Mean_Raw': clean_value(r[3]),
            'Duration_SD_Raw': clean_value(r[4]),
            'Duration_Min_Days': clean_value(r[5]),
            'Duration_Most_Likely_Days': clean_value(r[6]),
            'Duration_Max_Days': clean_value(r[7]),
            'Slack_Raw': clean_value(r[8]),
            'Variable_Cost_per_Day_x1000': clean_value(r[9]),
            'Fixed_Cost_x1000': clean_value(r[10]),
            'Cost_SD_Raw': clean_value(r[11]),
            'Cost_Min_Raw': clean_value(r[12]),
            'Cost_Most_Likely_Raw': clean_value(r[13]),
            'Cost_Max_Raw': clean_value(r[14]),
            'Cost_Model_Code': clean_value(r[15]),
            'Cost_Model': clean_value(r[16]),
            'Source_Excel_Row': excel_row,
        })
    return pd.DataFrame(rows)


def extract_risk_distribution_rows(data):
    distributions = {}
    # The source stores each risk in two rows: one probability row and one impact row.
    risk_area = data.iloc[40:71, 0:11].reset_index(drop=True)
    for i in range(1, len(risk_area), 2):
        p = risk_area.iloc[i]
        if i + 1 >= len(risk_area):
            break
        im = risk_area.iloc[i + 1]
        risk_id = clean_value(p[10])
        if not isinstance(risk_id, str) or not re.fullmatch(r'R\d+', risk_id):
            continue
        distributions[risk_id] = {
            'Probability_Distribution': clean_value(p[1]),
            'Probability_Model_Code': clean_value(p[2]),
            'Probability_Min': clean_value(p[5]),
            'Probability_Most_Likely_Raw': clean_value(p[6]),
            'Probability_Max': clean_value(p[7]),
            'Probability_Row_Type': clean_value(p[8]),
            'Risk_Target_Type_Raw': clean_value(p[9]),
            'Impact_Distribution': clean_value(im[1]),
            'Impact_Model_Code': clean_value(im[2]),
            'Impact_Min': clean_value(im[5]),
            'Impact_Most_Likely_Raw': clean_value(im[6]),
            'Impact_Max': clean_value(im[7]),
            'Impact_Row_Type': clean_value(im[8]),
            'Impact_Target_Type_Raw': clean_value(im[9]),
            'Raw_Data_Activity_Value': clean_value(im[10]),
            'Probability_Source_Row_Index': int(i + 41),
            'Impact_Source_Row_Index': int(i + 42),
        }
    return distributions


def extract_risk_table(risk_id_sheet, distributions):
    rows = []
    source_rows = risk_id_sheet.iloc[3:14, 0:11]
    for _, r in source_rows.iterrows():
        risk_name = clean_value(r[2])
        if risk_name is None:
            continue
        phase = clean_value(r[3])
        probability_level = clean_value(r[4])
        cost_code_raw = clean_value(r[0])
        schedule_code = clean_value(r[1])
        cost_impact_level = clean_value(r[5])
        cost_impact_interval = clean_value(r[6])
        time_impact_level = clean_value(r[7])
        time_impact_interval = clean_value(r[8])
        activity_number_published = clean_value(r[9])
        explanation = clean_value(r[10])

        # Schedule-risk row, if present.
        if isinstance(schedule_code, str) and re.fullmatch(r'R\d+', schedule_code):
            dist = distributions.get(schedule_code, {})
            min_pub, max_pub = split_interval(time_impact_interval)
            rows.append({
                'Risk_ID': schedule_code,
                'Risk_Type': 'Schedule',
                'Source_Risk_Name': risk_name,
                'Phase': phase,
                'Affected_Activity_ID': f'A{activity_number_published}' if activity_number_published is not None else None,
                'Affected_Activity_Number': activity_number_published,
                'Probability_Level': probability_level,
                'Probability_Level_Interval': PROBABILITY_LEVEL_INTERVAL.get(str(probability_level)),
                'Probability_Distribution': dist.get('Probability_Distribution'),
                'Probability_Model_Code': dist.get('Probability_Model_Code'),
                'Probability_Min': dist.get('Probability_Min'),
                'Probability_Max': dist.get('Probability_Max'),
                'Severity_Level': time_impact_level,
                'Severity_Level_Note': SEVERITY_LABEL_NOTE.get(str(time_impact_level), ''),
                'Severity_Interval_Published': time_impact_interval,
                'Severity_Min_Published': min_pub,
                'Severity_Max_Published': max_pub,
                'Impact_Distribution': dist.get('Impact_Distribution'),
                'Impact_Model_Code': dist.get('Impact_Model_Code'),
                'Impact_Min': dist.get('Impact_Min'),
                'Impact_Max': dist.get('Impact_Max'),
                'Impact_Units': 'days',
                'Raw_Data_Activity_Value': dist.get('Raw_Data_Activity_Value'),
                'Raw_Target_Type': dist.get('Risk_Target_Type_Raw'),
                'Explanation': explanation,
                'Source_Cost_Risk_Code_Raw': None,
                'Source_Schedule_Risk_Code_Raw': schedule_code,
                'Probability_Source_Row_Index': dist.get('Probability_Source_Row_Index'),
                'Impact_Source_Row_Index': dist.get('Impact_Source_Row_Index'),
            })

        # Cost-risk row, if present.
        if isinstance(cost_code_raw, str) and cost_code_raw in COST_RISK_ID_MAP:
            cost_risk_id = COST_RISK_ID_MAP[cost_code_raw]
            dist = distributions.get(cost_risk_id, {})
            min_pub, max_pub = split_interval(cost_impact_interval)
            rows.append({
                'Risk_ID': cost_risk_id,
                'Risk_Type': 'Cost',
                'Source_Risk_Name': risk_name,
                'Phase': phase,
                'Affected_Activity_ID': f'A{activity_number_published}' if activity_number_published is not None else None,
                'Affected_Activity_Number': activity_number_published,
                'Probability_Level': probability_level,
                'Probability_Level_Interval': PROBABILITY_LEVEL_INTERVAL.get(str(probability_level)),
                'Probability_Distribution': dist.get('Probability_Distribution'),
                'Probability_Model_Code': dist.get('Probability_Model_Code'),
                'Probability_Min': dist.get('Probability_Min'),
                'Probability_Max': dist.get('Probability_Max'),
                'Severity_Level': cost_impact_level,
                'Severity_Level_Note': SEVERITY_LABEL_NOTE.get(str(cost_impact_level), ''),
                'Severity_Interval_Published': cost_impact_interval,
                'Severity_Min_Published': min_pub,
                'Severity_Max_Published': max_pub,
                'Impact_Distribution': dist.get('Impact_Distribution'),
                'Impact_Model_Code': dist.get('Impact_Model_Code'),
                'Impact_Min': dist.get('Impact_Min'),
                'Impact_Max': dist.get('Impact_Max'),
                'Impact_Units': 'x1000 monetary units',
                'Raw_Data_Activity_Value': dist.get('Raw_Data_Activity_Value'),
                'Raw_Target_Type': dist.get('Risk_Target_Type_Raw'),
                'Explanation': explanation,
                'Source_Cost_Risk_Code_Raw': cost_code_raw,
                'Source_Schedule_Risk_Code_Raw': schedule_code if isinstance(schedule_code, str) else None,
                'Probability_Source_Row_Index': dist.get('Probability_Source_Row_Index'),
                'Impact_Source_Row_Index': dist.get('Impact_Source_Row_Index'),
            })
    risks = pd.DataFrame(rows)
    risks['Risk_Number'] = risks['Risk_ID'].str.extract(r'R(\d+)').astype(int)
    return risks.sort_values('Risk_Number').drop(columns=['Risk_Number']).reset_index(drop=True)


def attach_risks_to_activities(activities, risks):
    schedule_map = risks[risks['Risk_Type'].eq('Schedule')].groupby('Affected_Activity_ID')['Risk_ID'].apply(lambda s: ', '.join(s)).to_dict()
    cost_map = risks[risks['Risk_Type'].eq('Cost')].groupby('Affected_Activity_ID')['Risk_ID'].apply(lambda s: ', '.join(s)).to_dict()
    risk_name_map = risks.groupby('Affected_Activity_ID').apply(
        lambda g: '; '.join([f"{row.Risk_ID}: {row.Source_Risk_Name}" for row in g.itertuples()])
    ).to_dict()
    activities = activities.copy()
    activities['Schedule_Risk_IDs'] = activities['Activity_ID'].map(schedule_map).fillna('')
    activities['Cost_Risk_IDs'] = activities['Activity_ID'].map(cost_map).fillna('')
    activities['All_Connected_Risk_IDs'] = activities.apply(
        lambda r: ', '.join([x for x in [r['Schedule_Risk_IDs'], r['Cost_Risk_IDs']] if x]), axis=1
    )
    activities['Connected_Risk_Names'] = activities['Activity_ID'].map(risk_name_map).fillna('')
    activities['Activity_Duration_Uncertainty'] = activities.apply(
        lambda r: f"{r['Duration_Distribution']}({r['Duration_Min_Days']}, {r['Duration_Most_Likely_Days']}, {r['Duration_Max_Days']}) days",
        axis=1,
    )
    activities['Activity_Cost_Model'] = activities.apply(
        lambda r: f"Fixed {r['Fixed_Cost_x1000']} + {r['Variable_Cost_per_Day_x1000']} * duration, in x1000 monetary units",
        axis=1,
    )
    # Keep readable ordering.
    first_cols = [
        'Activity_ID','Activity_Number','Phase','Predecessors',
        'Schedule_Risk_IDs','Cost_Risk_IDs','All_Connected_Risk_IDs','Connected_Risk_Names',
        'Duration_Distribution','Duration_Min_Days','Duration_Most_Likely_Days','Duration_Max_Days',
        'Activity_Duration_Uncertainty','Fixed_Cost_x1000','Variable_Cost_per_Day_x1000','Activity_Cost_Model'
    ]
    rest = [c for c in activities.columns if c not in first_cols]
    return activities[first_cols + rest]


def build_clean_tables(input_path=INPUT):
    wb = load_source_workbook(input_path)
    activities = extract_raw_activity_table(wb['Data'])
    distributions = extract_risk_distribution_rows(wb['Data'])
    risks = extract_risk_table(wb['Risk Id'], distributions)
    activities = attach_risks_to_activities(activities, risks)
    return activities, risks


def write_clean_excel(activities, risks, output_path=OUTPUT_XLSX):
    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        activities.to_excel(writer, sheet_name='Activities', index=False)
        risks.to_excel(writer, sheet_name='Risks', index=False)
        workbook = writer.book
        header_fmt = workbook.add_format({'bold': True, 'bg_color': '#1F4E78', 'font_color': 'white', 'border': 1, 'text_wrap': True, 'valign': 'top'})
        text_wrap = workbook.add_format({'text_wrap': True, 'valign': 'top'})
        number_fmt = workbook.add_format({'num_format': '0.00', 'valign': 'top'})
        integer_fmt = workbook.add_format({'num_format': '0', 'valign': 'top'})
        for sheet_name, df in [('Activities', activities), ('Risks', risks)]:
            ws = writer.sheets[sheet_name]
            ws.freeze_panes(1, 0)
            ws.autofilter(0, 0, len(df), len(df.columns)-1)
            for col_idx, col in enumerate(df.columns):
                ws.write(0, col_idx, col, header_fmt)
                sample_values = [str(v) for v in df[col].dropna().head(50).tolist()]
                width = min(max([len(col)] + [len(v) for v in sample_values]) + 2, 42)
                if col in {'Explanation', 'Connected_Risk_Names', 'Activity_Duration_Uncertainty', 'Activity_Cost_Model'}:
                    width = 42
                ws.set_column(col_idx, col_idx, width, text_wrap)
            ws.set_row(0, 32)
        # Optional but useful: highlight the two requested connection columns.
        activities_ws = writer.sheets['Activities']
        for col_name in ['Schedule_Risk_IDs', 'Cost_Risk_IDs']:
            col_idx = activities.columns.get_loc(col_name)
            activities_ws.set_column(col_idx, col_idx, 18, text_wrap)
        risks_ws = writer.sheets['Risks']
        for col_name in ['Probability_Min', 'Probability_Max', 'Impact_Min', 'Impact_Max']:
            col_idx = risks.columns.get_loc(col_name)
            risks_ws.set_column(col_idx, col_idx, 14, number_fmt)
    return output_path



## Build the two cleaned tables

In [ ]:
activities, risks = build_clean_tables(INPUT)

print(f"Activities table: {activities.shape[0]} rows x {activities.shape[1]} columns")
print(f"Risks table: {risks.shape[0]} rows x {risks.shape[1]} columns")

activities.head()

## Check the activity-to-risk connections

In [ ]:
activities[['Activity_ID', 'Predecessors', 'Schedule_Risk_IDs', 'Cost_Risk_IDs', 'All_Connected_Risk_IDs', 'Connected_Risk_Names']]

## Check the cleaned risk table

In [ ]:
risks[['Risk_ID', 'Risk_Type', 'Source_Risk_Name', 'Affected_Activity_ID',
       'Probability_Level', 'Probability_Distribution', 'Probability_Min', 'Probability_Max',
       'Severity_Level', 'Impact_Distribution', 'Impact_Min', 'Impact_Max', 'Impact_Units']]

## Write the cleaned Excel workbook

In [ ]:
output_path = write_clean_excel(activities, risks, OUTPUT_XLSX)
print(f"Wrote cleaned workbook to: {output_path}")

## Validation checks

In [ ]:
assert len(activities) == 32, 'Expected 32 real activities A1:A32.'
assert len(risks) == 15, 'Expected 15 risk IDs R1:R15.'
assert set(['Schedule_Risk_IDs', 'Cost_Risk_IDs']).issubset(activities.columns)
assert set(['Risk_Type', 'Probability_Min', 'Probability_Max', 'Impact_Min', 'Impact_Max']).issubset(risks.columns)

print('Validation passed.')